# 3-Agent Melody Harmonization — Hub-and-Spoke Architecture

This notebook demonstrates a **3-agent pipeline** where an Orchestrator
coordinates iterative refinement between a Harmonizer and a Theory critic.

```
                 ┌──────────────────────┐
                 │    Orchestrator       │
                 │    (GPT-4o)           │
                 │    decides: APPROVE   │
                 │    or REVISE          │
                 └───┬──────────────┬────┘
                     │              │
            generate/│              │critique
             revise  │              │
                     ▼              ▼
           ┌──────────────┐  ┌──────────────┐
           │  Harmonizer  │  │  Theory      │
           │  (GPT-4o)    │  │  (Claude)    │
           │              │  │              │
           │  Generates   │  │  Critiques   │
           │  ABC chords  │  │  w/ OMT ctx  │
           │  No textbook │  │  No music gen│
           └──────────────┘  └──────────────┘
```

**Key design**: The Harmonizer has **no music theory textbook** — it relies
on its training and musical instinct. The Theory Agent has **all the Open
Music Theory context** but **never generates music** — it only critiques.
The Orchestrator mediates every exchange and decides when to stop.

## 1. Setup

In [ ]:
import sys
import pathlib
import tempfile
import os
import importlib

from dotenv import load_dotenv
from IPython.display import display, HTML

# Notebook-relative paths — auto-adapts when this folder is copied
notebook_dir = pathlib.Path.cwd()      # .../basic_agent_framework/<experiment>
parent_dir   = notebook_dir.parent     # .../basic_agent_framework
project_root = parent_dir.parent       # project root

EXPERIMENT_NAME = notebook_dir.name    # 'base', 'rag_experiment', etc.

sys.path.insert(0, str(project_root))  # for util
sys.path.insert(0, str(parent_dir))    # for package-style import of this experiment

load_dotenv(project_root / ".env")

for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY"]:
    assert os.getenv(key), f"{key} not set — add it to .env"

_exp = importlib.import_module(EXPERIMENT_NAME)
harmonize_melody             = _exp.harmonize_melody
load_bach_melody             = _exp.load_bach_melody
build_harmonization_template = _exp.build_harmonization_template
clean_abc_for_llm            = _exp.clean_abc_for_llm
AVAILABLE_BWV                = _exp.AVAILABLE_BWV

from util import abc_sonify as abc

SF2_PATH = str(project_root / "data" / "soundfonts" / "GeneralUser_GS.sf2")

print(f"Experiment : {EXPERIMENT_NAME}")
print("Setup complete!")
print(f"Available chorales: {AVAILABLE_BWV}")


## 2. Configuration

In [ ]:
# ── Chorale selection ────────────────────────────────────────────────────────
BWV = "bwv253"

# ── Model configuration ───────────────────────────────────────────────────────
ORCHESTRATOR_MODEL = "gpt-4o"
THEORY_MODEL       = "claude-sonnet-4-6"
HARMONIZER_MODEL   = "claude-sonnet-4-6"
MAX_ITERATIONS     = 3

# ── Audio output directory ────────────────────────────────────────────────────
model_tag     = f"{ORCHESTRATOR_MODEL}_{THEORY_MODEL}_{HARMONIZER_MODEL}"
audio_out_dir = parent_dir / "experiments" / EXPERIMENT_NAME / model_tag
audio_out_dir.mkdir(parents=True, exist_ok=True)
print(f"Audio output: {audio_out_dir}")

# ── Audio save helper (soundfile preferred, scipy fallback) ───────────────────
try:
    import soundfile as _sf
    def _save_wav(path, audio, sr):
        _sf.write(str(path), audio, sr)
    print("WAV export: soundfile")
except ImportError:
    import numpy as _np
    from scipy.io import wavfile as _wavfile
    def _save_wav(path, audio, sr):
        audio_i16 = _np.clip(audio * 32767, -32768, 32767).astype(_np.int16)
        _wavfile.write(str(path), sr, audio_i16)
    print("WAV export: scipy")

print(f"Models: orch={ORCHESTRATOR_MODEL} | theory={THEORY_MODEL} | harm={HARMONIZER_MODEL}")


## 3. Load a Bach Melody

In [ ]:
single_voice = load_bach_melody(BWV, measures=(1, 8))
template     = build_harmonization_template(single_voice, title_override=f"{BWV.upper()} Soprano")
melody_abc   = clean_abc_for_llm(template)

print(melody_abc)


### Listen to the raw melody

In [ ]:
with tempfile.NamedTemporaryFile(mode="w", suffix=".abc", delete=False, encoding="utf-8") as tmp:
    tmp.write(melody_abc)
    melody_tmp = pathlib.Path(tmp.name)

try:
    score = abc.load_abc(melody_tmp)
    display(HTML("<b>Soprano melody (V:1 only):</b>"))
    audio, sr = abc.sonify_part(score, 0, sf2_path=SF2_PATH)
    display(abc.play_audio(audio, sr))
    _save_wav(audio_out_dir / f"{BWV}_melody.wav", audio, sr)
    print(f"Saved: {BWV}_melody.wav")
finally:
    melody_tmp.unlink(missing_ok=True)


## 4. Run the Hub-and-Spoke Pipeline

The pipeline runs up to `max_iterations` rounds of:
1. Harmonizer generates/revises chords
2. Theory Agent critiques the result
3. Orchestrator decides: **APPROVED** or **REVISE**

Watch the printed status to see the agents coordinating.

In [ ]:
print("Running hub-and-spoke pipeline...\n")

result = await harmonize_melody(
    melody_abc,
    max_iterations=MAX_ITERATIONS,
    orchestrator_model=ORCHESTRATOR_MODEL,
    theory_model=THEORY_MODEL,
    harmonizer_model=HARMONIZER_MODEL,
    verbose=True,
)

print(f"\nDone! {len(result.iterations)} iteration(s), "
      f"final status: {'APPROVED' if result.iterations[-1].approved else 'MAX ITERATIONS'}")


## 5. Inspect Each Iteration

The `HarmonizationResult` stores every round so you can see how the
Harmonizer's output evolved based on the Theory Agent's feedback.

In [ ]:
import difflib
from html import escape

_pre = (
    "padding:10px; font-size:12px; color:#111827; border:1px solid #64748b; "
    "border-radius:4px; white-space:pre-wrap; overflow-x:auto;"
)

prev_abc = None
for it in result.iterations:
    status = "APPROVED" if it.approved else "REVISE"
    color = "#2e7d32" if it.approved else "#c62828"

    display(HTML(
        f"<h3 style='color:{color}'>Iteration {it.attempt} — {status}</h3>"
    ))

    display(HTML("<b>Harmonizer's output (full ABC):</b>"))
    print(f"\n### Iteration {it.attempt} ({status}) — full ABC ###\n")
    print(it.harmonization)
    display(HTML(f"<pre style='background:#f0f8ff; {_pre}'>{escape(it.harmonization)}</pre>"))

    if prev_abc is not None:
        diff_lines = list(
            difflib.unified_diff(
                prev_abc.splitlines(),
                it.harmonization.splitlines(),
                fromfile=f"round_{it.attempt - 1}",
                tofile=f"round_{it.attempt}",
                lineterm="",
            )
        )
        diff_text = "\n".join(diff_lines)
        display(HTML("<b>ABC changes vs previous round (unified diff):</b>"))
        print("\n--- unified diff vs previous round ---\n")
        print(diff_text if diff_text else "(no line changes)")
        display(HTML(
            f"<pre style='background:#eceff1; {_pre}'>{escape(diff_text or '(no line changes)')}</pre>"
        ))

    prev_abc = it.harmonization

    display(HTML("<b>Theory Agent's critique:</b>"))
    display(HTML(f"<pre style='background:#fff8e1; {_pre}'>{escape(it.critique)}</pre>"))

    display(HTML("<b>Orchestrator's decision:</b>"))
    display(HTML(f"<pre style='background:#f3e5f5; {_pre}'>{escape(it.decision)}</pre>"))

    display(HTML("<hr>"))


## 6. Final Harmonized ABC

In [ ]:
display(HTML("<h3>Final 2-Voice ABC</h3>"))
display(HTML(
    "<pre style='background:#e8f5e9; padding:12px; color:#111827; border:1px solid #64748b; border-radius:4px'>"
    f"{result.final_abc}</pre>"
))

## 7. Listen to the Result

In [ ]:
with tempfile.NamedTemporaryFile(mode="w", suffix=".abc", delete=False, encoding="utf-8") as tmp:
    tmp.write(result.final_abc)
    result_tmp = pathlib.Path(tmp.name)

try:
    score = abc.load_abc(result_tmp)
    parts = abc.list_parts(score)

    for p in parts:
        t0 = p.get("start_time", 0)
        color = "green" if t0 < 0.5 else "red"
        display(HTML(
            f'<small style="color:{color}">'
            f'[{p["instrument_index"]}] {p["name"]} — '
            f'start: {t0:.2f}s, end: {p.get("end_time", "?"):.2f}s'
            f'</small><br>'
        ))

    display(HTML("<b>Melody only (V:1):</b>"))
    melody_audio, sr = abc.sonify_part(score, 0, sf2_path=SF2_PATH)
    display(abc.play_audio(melody_audio, sr))

    display(HTML("<b>Melody + Chords:</b>"))
    mix_audio, sr = abc.sonify_parts(score, [0, 1], sf2_path=SF2_PATH)
    display(abc.play_audio(mix_audio, sr))

    try:
        display(HTML("<b>Chords only (V:2):</b>"))
        chords_audio, sr = abc.sonify_part(score, 1, sf2_path=SF2_PATH)
        display(abc.play_audio(chords_audio, sr))
    except Exception as e:
        display(HTML(f'<p style="color:orange">Could not isolate chords: {e}</p>'))

    meta = abc.get_metadata(score)
    display(HTML(
        f"<small>Key: {meta['key']} | Tempo: {meta['tempo_bpm']} BPM | "
        f"Parts: {meta['num_parts']} | Duration: {meta['duration_seconds']:.1f}s</small>"
    ))

except RuntimeError as e:
    display(HTML(f'<p style="color:red"><b>abc2midi parse error:</b> {e}</p>'))
    display(HTML(f"<pre>{result.final_abc}</pre>"))

finally:
    result_tmp.unlink(missing_ok=True)


## 8. Compare Iterations (per round)

For **each** round: unified diff vs the previous round (when applicable), full ABC below, `print` in the console, and three audio players (melody + chords, melody only, chords only). Audio is also saved to the `experiments/` folder.

In [ ]:
import difflib
from html import escape

_pre = (
    "padding:10px; font-size:12px; color:#111827; border:1px solid #64748b; "
    "border-radius:4px; white-space:pre-wrap; overflow-x:auto;"
)


def _play_three_audios(abc_text: str, title: str, save_prefix: str = None) -> None:
    """Sonify one iteration: melody+chords, melody only, chords only.
    If save_prefix is given, also save WAV files to audio_out_dir.
    """
    with tempfile.NamedTemporaryFile(mode="w", suffix=".abc", delete=False, encoding="utf-8") as tmp:
        tmp.write(abc_text)
        p = pathlib.Path(tmp.name)
    try:
        score = abc.load_abc(p)
        mix_audio, sr = abc.sonify_parts(score, [0, 1], sf2_path=SF2_PATH)
        display(HTML(f"<b>{escape(title)} — Melody + Chords</b>"))
        display(abc.play_audio(mix_audio, sr))
        if save_prefix:
            _save_wav(audio_out_dir / f"{save_prefix}_mix.wav", mix_audio, sr)

        mel_audio, sr = abc.sonify_part(score, 0, sf2_path=SF2_PATH)
        display(HTML(f"<b>{escape(title)} — Melody only (V:1)</b>"))
        display(abc.play_audio(mel_audio, sr))
        if save_prefix:
            _save_wav(audio_out_dir / f"{save_prefix}_melody.wav", mel_audio, sr)

        ch_audio, sr = abc.sonify_part(score, 1, sf2_path=SF2_PATH)
        display(HTML(f"<b>{escape(title)} — Chords only (V:2)</b>"))
        display(abc.play_audio(ch_audio, sr))
        if save_prefix:
            _save_wav(audio_out_dir / f"{save_prefix}_chords.wav", ch_audio, sr)
    except Exception as e:
        display(HTML(f"<p style='color:red'><b>Audio error ({escape(title)}):</b> {escape(str(e))}</p>"))
    finally:
        p.unlink(missing_ok=True)


prev = None
for it in result.iterations:
    status = "APPROVED" if it.approved else "REVISE"
    display(HTML(f"<h3>Round {it.attempt} — {escape(status)}</h3>"))

    print(f"\n{'='*60}\nRound {it.attempt} ({status}) — full ABC\n{'='*60}\n")
    print(it.harmonization)

    display(HTML("<b>Full ABC (this round)</b>"))
    display(HTML(f"<pre style='background:#e8eaf6; {_pre}'>{escape(it.harmonization)}</pre>"))

    if prev is not None:
        diff_lines = list(
            difflib.unified_diff(
                prev.splitlines(),
                it.harmonization.splitlines(),
                fromfile=f"round_{it.attempt - 1}",
                tofile=f"round_{it.attempt}",
                lineterm="",
            )
        )
        diff_text = "\n".join(diff_lines)
        print(f"\n--- Diff vs round {it.attempt - 1} ---\n")
        print(diff_text if diff_text else "(no line changes)")
        display(HTML("<b>Unified diff vs previous round (ABC text)</b>"))
        display(HTML(f"<pre style='background:#fff3e0; {_pre}'>{escape(diff_text or '(no line changes)')}</pre>"))

    _play_three_audios(
        it.harmonization,
        f"Round {it.attempt}",
        save_prefix=f"{BWV}_iter{it.attempt}",
    )
    prev = it.harmonization
    display(HTML("<hr>"))

if len(result.iterations) == 1:
    display(HTML("<small>Only one round ran; diff blocks appear when the Harmonizer revises.</small>"))


## 9. Save Iteration Log

Writes a single text file containing every iteration's ABC + the final ABC to
the same folder as the audio files. Use this for cross-experiment comparison.

In [ ]:
iterations_txt_path = audio_out_dir / f"{BWV}_iterations.txt"

with open(iterations_txt_path, "w", encoding="utf-8") as f:
    f.write(f"# Harmonization Log — {BWV}\n")
    f.write(f"# Experiment:    {EXPERIMENT_NAME}\n")
    f.write(f"# Orchestrator:  {ORCHESTRATOR_MODEL}\n")
    f.write(f"# Theory:        {THEORY_MODEL}\n")
    f.write(f"# Harmonizer:    {HARMONIZER_MODEL}\n")
    f.write(f"# Iterations:    {len(result.iterations)}\n\n")

    for it in result.iterations:
        status = "APPROVED" if it.approved else "REVISE"
        f.write("=" * 80 + "\n")
        f.write(f"ITERATION {it.attempt} — {status}\n")
        f.write("=" * 80 + "\n\n")

        reasoning = getattr(it, "harmonizer_reasoning", "")
        if reasoning:
            f.write("--- Harmonizer reasoning ---\n")
            f.write(reasoning.rstrip() + "\n\n")

        f.write("--- ABC ---\n")
        f.write(it.harmonization.rstrip() + "\n\n")

    f.write("=" * 80 + "\n")
    f.write("FINAL ABC\n")
    f.write("=" * 80 + "\n\n")
    f.write(result.final_abc.rstrip() + "\n")

print(f"Saved iteration log: {iterations_txt_path}")
